In [6]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os
import certifi
import time

# Force TLS to use certifi bundle (macOS fix)
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()

# Resolve project root when running from Our Notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
!pip install numpy pandas xarray scipy tqdm pystac-client planetary-computer zarr fsspec adlfs

In [2]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [3]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final


In [4]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

In [ ]:
# Chunked TerraClimate extraction (point-only, no full-grid cache)
# Uses nearest neighbor lookups directly on the grid for each sample.

tc_vars = [
    'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'swe',
    'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi'
]

chunk_size = 200
max_retries = 3
retry_wait_seconds = 60

train_out = os.path.join(PROJECT_ROOT, 'New Datasets', 'terraclimate_features_training_allvars.csv')
val_out = os.path.join(PROJECT_ROOT, 'New Datasets', 'terraclimate_features_validation_allvars.csv')

expected_cols = ['Latitude', 'Longitude', 'Sample Date'] + tc_vars


def count_rows_in_csv(path: str) -> int:
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


def select_points(ds, var: str, df: pd.DataFrame) -> np.ndarray:
    times = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
    times_filled = times.fillna(pd.Timestamp('2011-01-01'))
    lats = df['Latitude'].values
    lons = df['Longitude'].values

    points = xr.Dataset({
        'lat': (('points',), lats),
        'lon': (('points',), lons),
        'time': (('points',), times_filled.values),
    })

    vals = ds[var].sel(
        lat=points['lat'],
        lon=points['lon'],
        time=points['time'],
        method='nearest',
    ).values.astype(float)

    mask = times.isna() | pd.isna(lats) | pd.isna(lons)
    vals[mask.values] = np.nan
    return vals


def extract_chunked(input_df: pd.DataFrame, output_path: str, label: str):
    start_idx = 0
    if os.path.exists(output_path):
        existing_header = pd.read_csv(output_path, nrows=0).columns.tolist()
        if existing_header != expected_cols:
            raise ValueError(
                f"Existing file has different columns.\n"
                f"Expected: {expected_cols}\n"
                f"Found:    {existing_header}\n"
                f"Fix: delete/rename the existing file or update expected_cols."
            )
        start_idx = count_rows_in_csv(output_path)

    print(f"🚀 Running TerraClimate extraction for {label} (chunked)...")
    print(f"Total rows: {len(input_df)}")
    print(f"Output file: {output_path}")
    print(f"Chunk size: {chunk_size}")
    print(f"Resuming from row index: {start_idx}")

    ds = load_terraclimate_dataset()

    for chunk_start in range(start_idx, len(input_df), chunk_size):
        chunk_end = min(chunk_start + chunk_size, len(input_df))
        chunk_df = input_df.iloc[chunk_start:chunk_end].copy()
        print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")

        try:
            chunk_out = chunk_df[['Latitude', 'Longitude', 'Sample Date']].copy()

            for var in tc_vars:
                for attempt in range(1, max_retries + 1):
                    try:
                        chunk_out[var] = select_points(ds, var, chunk_df)
                        break
                    except Exception as exc:
                        msg = str(exc)
                        if "AuthenticationFailed" in msg or "ClientAuthenticationError" in msg:
                            ds = load_terraclimate_dataset()
                        if attempt == max_retries:
                            raise
                        time.sleep(retry_wait_seconds)

            write_header = (not os.path.exists(output_path)) or (count_rows_in_csv(output_path) == 0)
            chunk_out.to_csv(output_path, mode='a', header=write_header, index=False)
            time.sleep(0.5)

        except Exception as exc:
            msg = str(exc)
            print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {msg}")
            print("You can rerun this cell to resume from the last completed chunk.")
            break


# Load data
Water_Quality_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'water_quality_training_dataset.csv'))
Validation_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'submission_template.csv'))

# Run extraction
extract_chunked(Water_Quality_df, train_out, "training")
extract_chunked(Validation_df, val_out, "validation")

# Preview
if os.path.exists(train_out):
    display(pd.read_csv(train_out).head())
if os.path.exists(val_out):
    display(pd.read_csv(val_out).head())